In [1]:
%load_ext autoreload
%autoreload 2
import sys
from pathlib import Path
path = str(Path.cwd().parent)
print(path)
sys.path.insert(1, path)

import numpy as np
import pandas as pd
import skforecast

print(skforecast.__version__)

c:\Users\Joaquin\Documents\GitHub\skforecast
0.25.0


In [2]:
# Libraries
# ==============================================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pprint import pprint
from skforecast.foundation import FoundationModel, ForecasterFoundation
from skforecast.plot import set_dark_theme
from skforecast.preprocessing import (
    reshape_series_long_to_dict, 
    reshape_exog_long_to_dict, 
    RollingFeatures
)
from skforecast.model_selection import (
    TimeSeriesFold,
    backtesting_foundation,
    bayesian_search_foundation
)

# Load time series of multiple lengths and exogenous variables
# ==============================================================================
series = pd.read_csv(
    'https://raw.githubusercontent.com/skforecast/skforecast-datasets/main/data/demo_multi_series.csv'
)
exog = pd.read_csv(
    'https://raw.githubusercontent.com/skforecast/skforecast-datasets/main/data/demo_multi_series_exog.csv'
)

series['timestamp'] = pd.to_datetime(series['timestamp'])
exog['timestamp'] = pd.to_datetime(exog['timestamp'])

display(series.head(3))
print("")
display(exog.head(3))


# Transform series and exog to dictionaries
# ==============================================================================
series_dict = reshape_series_long_to_dict(
    data      = series,
    series_id = 'series_id',
    index     = 'timestamp',
    values    = 'value',
    freq      = 'D'
)

exog_dict = reshape_exog_long_to_dict(
    data      = exog,
    series_id = 'series_id',
    index     = 'timestamp',
    freq      = 'D'
)


# Drop some exogenous variables for series 'id_1000' and 'id_1003'
# ==============================================================================
# Some exogenous variables are intentionally omitted for series 1 and 3 to illustrate that each series can use a different set of exogenous variables.
exog_dict['id_1000'] = exog_dict['id_1000'].drop(columns=['air_temperature', 'wind_speed'])
exog_dict['id_1003'] = exog_dict['id_1003'].drop(columns=['cos_day_of_week'])


# Partition data in train and test
# ==============================================================================
end_train = '2016-07-31 23:59:00'
series_dict_train = {k: v.loc[: end_train,] for k, v in series_dict.items()}
exog_dict_train   = {k: v.loc[: end_train,] for k, v in exog_dict.items()}
series_dict_test  = {k: v.loc[end_train:,] for k, v in series_dict.items()}
exog_dict_test    = {k: v.loc[end_train:,] for k, v in exog_dict.items()}


# Description of each partition
# ==============================================================================
for k in series_dict.keys():
    print(f"{k}:")
    try:
        print(
            f"\tTrain: len={len(series_dict_train[k])}, {series_dict_train[k].index[0]}"
            f" --- {series_dict_train[k].index[-1]}"
        )
    except IndexError:
        print("\tTrain: len=0")
    try:
        print(
            f"\tTest : len={len(series_dict_test[k])}, {series_dict_test[k].index[0]}"
            f" --- {series_dict_test[k].index[-1]}"
        )
    except IndexError:
        print("\tTest : len=0")


# Exogenous variables for each series
# ==============================================================================
for k in series_dict.keys():
    print(f"{k}:")
    try:
        print(f"\t{exog_dict[k].columns.to_list()}")
    except IndexError:
        print("\tNo exogenous variables")


# Fit and Predict forecaster
# ==============================================================================
model_ids = [
    "autogluon/chronos-2-small",
    "google/timesfm-3.0-pytorch",
    "google/timesfm-2.5-200m-pytorch",
    "soda-inria/tabicl",
    "priorlabs/tabpfn-ts",
    "theforecastingcompany/t0-alpha",
    "Synthefy/Nori",
    "taharnbl/TS-ICL"
]


for model_id in model_ids:
    try:
        estimator = FoundationModel(model_id=model_id, context_length=500)
        forecaster = ForecasterFoundation(estimator=estimator)
        forecaster.fit(series=series_dict_train, exog=exog_dict_train)
        predictions = forecaster.predict(steps=5, exog=exog_dict_test)
        print(predictions.head(9))
    except Exception as e:
        # Now you will know exactly which model threw the error
        print(f"FAILED - Error with {model_id}: {e}")



,series_id,timestamp,value
0,id_1000,2016-01-01,1012.500694
1,id_1000,2016-01-02,1158.500099
2,id_1000,2016-01-03,983.000099


,series_id,timestamp,sin_day_of_week,cos_day_of_week,air_temperature,wind_speed
0,id_1000,2016-01-01,-0.433884,-0.900969,6.416639,4.040115
1,id_1000,2016-01-02,-0.974928,-0.222521,6.366474,4.530395
2,id_1000,2016-01-03,-0.781831,0.623490,6.555272,3.273064


╭──────────────────────────────── MissingValuesWarning ────────────────────────────────╮
│ Series 'id_1003' is incomplete. NaNs have been introduced after setting the          │
│ frequency.                                                                           │
│                                                                                      │
│ Category : skforecast.exceptions.MissingValuesWarning                                │
│ Location :                                                                           │
│ c:\Users\Joaquin\Documents\GitHub\skforecast\skforecast\preprocessing\_preprocessing │
│ .py:531                                                                              │
│ Suppress : warnings.simplefilter('ignore', category=MissingValuesWarning)            │
╰──────────────────────────────────────────────────────────────────────────────────────╯

id_1000:
	Train: len=213, 2016-01-01 00:00:00 --- 2016-07-31 00:00:00
	Test : len=153, 2016-08-01 00:00:00 --- 2016-12-31 00:00:00
id_1001:
	Train: len=30, 2016-07-02 00:00:00 --- 2016-07-31 00:00:00
	Test : len=153, 2016-08-01 00:00:00 --- 2016-12-31 00:00:00
id_1002:
	Train: len=183, 2016-01-01 00:00:00 --- 2016-07-01 00:00:00
	Test : len=0
id_1003:
	Train: len=213, 2016-01-01 00:00:00 --- 2016-07-31 00:00:00
	Test : len=153, 2016-08-01 00:00:00 --- 2016-12-31 00:00:00
id_1004:
	Train: len=91, 2016-05-02 00:00:00 --- 2016-07-31 00:00:00
	Test : len=31, 2016-08-01 00:00:00 --- 2016-08-31 00:00:00
id_1000:
	['sin_day_of_week', 'cos_day_of_week']
id_1001:
	['sin_day_of_week', 'cos_day_of_week', 'air_temperature', 'wind_speed']
id_1002:
	['sin_day_of_week', 'cos_day_of_week', 'air_temperature', 'wind_speed']
id_1003:
	['sin_day_of_week', 'air_temperature', 'wind_speed']
id_1004:
	['sin_day_of_week', 'cos_day_of_week', 'air_temperature', 'wind_speed']


╭──────────────────────────────── MissingValuesWarning ────────────────────────────────╮
│ `exog` for series ['id_1002'] has been reindexed to match the expected forecast      │
│ horizon. Missing timestamps were filled with NaN.                                    │
│                                                                                      │
│ Category : skforecast.exceptions.MissingValuesWarning                                │
│ Location : C:\Users\Joaquin\AppData\Local\Temp\ipykernel_2828\2717422903.py:120      │
│ Suppress : warnings.simplefilter('ignore', category=MissingValuesWarning)            │
╰──────────────────────────────────────────────────────────────────────────────────────╯

...

╭──────────────────────────────── MissingValuesWarning ────────────────────────────────╮
│ `exog` for series ['id_1002'] has been reindexed to match the expected forecast      │
│ horizon. Missing timestamps were filled with NaN.                                    │
│                                                                                      │
│ Category : skforecast.exceptions.MissingValuesWarning                                │
│ Location : C:\Users\Joaquin\AppData\Local\Temp\ipykernel_2828\2717422903.py:120      │
│ Suppress : warnings.simplefilter('ignore', category=MissingValuesWarning)            │
╰──────────────────────────────────────────────────────────────────────────────────────╯

              level         pred
2016-08-01  id_1000  1367.172518
2016-08-01  id_1001  2488.907479
2016-07-02  id_1002  5649.109674
2016-08-01  id_1003  2749.975271
2016-08-01  id_1004  7322.082160
2016-08-02  id_1000  1419.453840
2016-08-02  id_1001  2455.196370
2016-07-03  id_1002  4930.685889
2016-08-02  id_1003  2197.021081


╭──────────────────────────────── MissingValuesWarning ────────────────────────────────╮
│ `exog` for series ['id_1002'] has been reindexed to match the expected forecast      │
│ horizon. Missing timestamps were filled with NaN.                                    │
│                                                                                      │
│ Category : skforecast.exceptions.MissingValuesWarning                                │
│ Location : C:\Users\Joaquin\AppData\Local\Temp\ipykernel_2828\2717422903.py:120      │
│ Suppress : warnings.simplefilter('ignore', category=MissingValuesWarning)            │
╰──────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────── LicenseWarning ───────────────────────────────────╮
│ The weights for 'taharnbl/TS-ICL' are released under tsicl-v1-license-v1.0           │
│ (non-commercial). Review the license terms before commercial or production use. See  │
│ https://huggingface.co/taharnbl/TS-ICL.                                              │
│                                                                                      │
│ Category : skforecast.exceptions.LicenseWarning                                      │
│ Location :                                                                           │
│ c:\Users\Joaquin\Documents\GitHub\skforecast\skforecast\foundation\_adapters.py:3891 │
│ Suppress : warnings.simplefilter('ignore', category=LicenseWarning)                  │
╰──────────────────────────────────────────────────────────────────────────────────────╯

              level         pred
2016-08-01  id_1000  1366.310181
2016-08-01  id_1001  2445.619141
2016-07-02  id_1002  6576.323730
2016-08-01  id_1003  2174.525391
2016-08-01  id_1004  6672.560059
2016-08-02  id_1000  1424.961182
2016-08-02  id_1001  2299.456543
2016-07-03  id_1002  5033.787109
2016-08-02  id_1003  1973.090942


In [ ]:
# Backtesting
# ==============================================================================
cv = TimeSeriesFold(
         steps              = 24,
         initial_train_size = "2016-07-31 23:59:00",
     )

for model_id in model_ids:
    print(f"Backtesting with model {model_id}")
    try:
        metrics_levels, backtest_predictions = backtesting_foundation(
            forecaster            = forecaster,
            series                = series_dict,
            exog                  = exog_dict,
            cv                    = cv,
            levels                = None,
            metric                = "mean_absolute_error",
            add_aggregated_metric = True,
            suppress_warnings     = True
        )
    except Exception as e:
        # Now you will know exactly which model threw the error
        print(f"FAILED - Error with {model_id}: {e}")

Backtesting with model autogluon/chronos-2-small


  0%|          | 0/7 [00:00<?, ?it/s]

Backtesting with model google/timesfm-3.0-pytorch


  0%|          | 0/7 [00:00<?, ?it/s]

Backtesting with model google/timesfm-2.5-200m-pytorch


...

  0%|          | 0/7 [00:00<?, ?it/s]

Backtesting with model Synthefy/Nori


  0%|          | 0/7 [00:00<?, ?it/s]

Backtesting with model taharnbl/TS-ICL


  0%|          | 0/7 [00:00<?, ?it/s]